In [425]:
import random as rd
from typing import Tuple

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd

from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.normalizers import Lowercase
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.decoders import BPEDecoder, Replace

from transformers import PreTrainedTokenizerFast

In [47]:
dataset = load_dataset("mteb/emotion")
print(f"Dataset type: {type(dataset)}")
dataset

Dataset type: <class 'datasets.dataset_dict.DatasetDict'>


DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 15956
    })
    validation: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1988
    })
    test: Dataset({
        features: ['text', 'label', 'label_text'],
        num_rows: 1986
    })
})

In [48]:
train_dataset = dataset["train"].to_pandas()
val_dataset = dataset["validation"].to_pandas()
test_dataset = dataset['test'].to_pandas()
print(f"Splits dtype: {type(train_dataset), type(val_dataset), type(train_dataset)}")
print(len(train_dataset), len(val_dataset), len(test_dataset))

Splits dtype: (<class 'pandas.DataFrame'>, <class 'pandas.DataFrame'>, <class 'pandas.DataFrame'>)
15956 1988 1986


In [64]:
print("Random train row:")
print(train_dataset.sample(), end = '\n\n')
print("Random val row:")
print(val_dataset.sample(), end = '\n\n')
print("Random test row:")
print(test_dataset.sample())

Random train row:
                                                    text  label label_text
10896  i feel that it took a lot of guts on her part ...      2       love

Random val row:
                                                   text  label label_text
1277  i wonder what the other students in my classes...      1        joy

Random test row:
                                                   text  label label_text
1921  ive started feeling like almost nothing is wor...      4       fear


In [355]:
# Create tokeinzer object from HF tokenizers
tokenizer = Tokenizer(model = BPE(unk_token="[UNK]"))

# Trainer tokenizer
trainer = BpeTrainer(
    vocab_size =2_000,  # Desired vocabulary size
    min_frequency= 2,   # Minimum frequency for a token to be included
    special_tokens=["[PAD]", "[UNK]", "[CLS]"]
)
tokenizer.normalizer = Lowercase()
tokenizer.decoder = BPEDecoder(suffix = "#")
tokenizer.pad_token = 0

In [356]:
train_dataset["text"]

0                                  i didnt feel humiliated
1        i can go from feeling so hopeless to so damned...
2         im grabbing a minute to post i feel greedy wrong
3        i am ever feeling nostalgic about the fireplac...
4                                     i am feeling grouchy
                               ...                        
15951    i just had a very brief time in the beanbag an...
15952    i am now turning and i feel pathetic that i am...
15953                       i feel strong and good overall
15954    i feel like this was such a rude comment and i...
15955    i know a lot but i feel so stupid because i ca...
Name: text, Length: 15956, dtype: str

In [357]:
tokenizer.train_from_iterator(train_dataset["text"], trainer = trainer)

In [358]:
tokenizer.get_vocab()

{'ated ': 324,
 'usually ': 1457,
 'are ': 194,
 'wonderful ': 1455,
 'fre': 527,
 'ther ': 209,
 'weird ': 1071,
 'i': 12,
 'class ': 1504,
 'help but ': 1221,
 'ton': 1289,
 'impres': 1381,
 'gent': 1896,
 'work ': 505,
 'through the ': 1752,
 'talk ': 1049,
 'the most ': 1431,
 'excited ': 1503,
 'job ': 1326,
 'k': 14,
 'realiz': 1825,
 'become ': 1247,
 'ak ': 1427,
 'moo': 1574,
 'inside ': 1331,
 'conf': 1227,
 'insul': 1631,
 'cep': 559,
 'front ': 1886,
 'rag': 1057,
 'admit ': 1481,
 'i can feel ': 1353,
 'make ': 331,
 'on and ': 1573,
 'hou': 1155,
 'vel ': 1796,
 'that im ': 1544,
 'lethargic ': 1788,
 'he': 386,
 'red ': 1949,
 'extremely ': 1310,
 'i feel more ': 1088,
 'thought ': 810,
 'vie': 1250,
 'told ': 1237,
 'gro': 1398,
 'back': 1763,
 'bitch': 1782,
 'ed to have ': 1958,
 'to a ': 1539,
 'ed my ': 1739,
 'whether ': 1761,
 'cra': 692,
 'mother ': 1517,
 'optimis': 1762,
 'somehow ': 1615,
 'cer': 619,
 'a lot of ': 999,
 'of being ': 1359,
 'l ': 66,
 'fant': 

In [362]:
main_tokenizer = PreTrainedTokenizerFast(tokenizer_object = tokenizer, pad_token = "[PAD]")

In [ ]:
input_ids = main_tokenizer("never you mind that weakling", padding = "max_length", max_length = 50, trucation = True)['input_ids']
input_ids

[539,
 167,
 1100,
 70,
 1960,
 14,
 15,
 42,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [365]:
max_number_of_tokens = 0
for text in train_dataset["text"]:
    tokenized_text = main_tokenizer(text)['input_ids']
    max_number_of_tokens = max(max_number_of_tokens, len(tokenized_text))
print(f"Maxium number of tokens in a single sentence: {max_number_of_tokens}")

Maxium number of tokens in a single sentence: 108


In [368]:
main_tokenizer.decode(input_ids, skip_special_tokens = True)

'never you mind that weakling'

In [410]:
# Create torch datasets
class EmotionDataset(Dataset):
    def __init__(self, data: pd.DataFrame, 
                 tokenizer: PreTrainedTokenizerFast, max_length: int = 108):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __getitem__(self, idx) -> Tuple[torch.tensor, int]:
        single_row = self.data.iloc[idx]
        raw_text = single_row["text"]
        tokenized_text = self.tokenizer(raw_text, padding = "max_length",
                                        max_length = self.max_length, truncation = True,
                                        return_tensors = "pt")["input_ids"]
        return tokenized_text, single_row["label"].item()

    def __len__(self):
        return len(self.data)


In [411]:
train_dataset.iloc[0]

text          i didnt feel humiliated
label                               0
label_text                    sadness
Name: 0, dtype: object

In [462]:
# Random emotions dataset item
emotion_dataset = EmotionDataset(train_dataset, tokenizer = main_tokenizer)
random_item = emotion_dataset[rd.randint(0, len(emotion_dataset) - 1)]
print(f"Random item tensor shape: {random_item[0].shape}")
random_item

Random item tensor shape: torch.Size([1, 108])


(tensor([[ 405, 1193, 1407,  111,   33,  312,   31,    6,  669,  712, 1805,  316,
           403,  455,  196,   54,  246,  793,  332,   28,  525,  183,   15,   64,
          1350, 1015,  130,  429,  103,   17,   10, 1146,   74,  738,  365,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
             0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0]]),
 1)

In [468]:
# Prepare all the other datasets
emotions_dataset_train = EmotionDataset(train_dataset, tokenizer = main_tokenizer)
emotions_dataset_val = EmotionDataset(val_dataset, tokenizer = main_tokenizer)
emotions_dataset_test = EmotionDataset(test_dataset, tokenizer = main_tokenizer)
print(f"Lengths of datasets: {len(emotions_dataset_train)}, {len(emotions_dataset_val)}, {len(emotions_dataset_test)}")


Lengths of datasets: 15956, 1988, 1986
